<a href="https://colab.research.google.com/github/tousif47/Geostormcaster/blob/main/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data Preparation: Model A (t+3h Kp forecast)

Builds training/validation/test windows from `omni_clean.csv`.

- **Sample** = 24h solar wind window (5 features) ending at hour *t*, labeled with the Kp of the *next* 3-hour block
- **Splits** = time-ordered: train ≤2010, val 2011–2018, test 2019–present (random splits would leak the solar cycle)
- **Output** = `windows_a.npz` + `windows_a_norm.json` (normalization uses train statistics only)

In [ ]:
import pandas as pd
import numpy as np
import json
from numpy.lib.stride_tricks import sliding_window_view

CSV     = '/content/drive/MyDrive/temp/omni_clean.csv'
OUT_DIR = '/content/drive/MyDrive/temp'

## Load and clean

In [ ]:
df = pd.read_csv(CSV, parse_dates=['timestamp']).set_index('timestamp').sort_index()
df['kp'] = df['kp'].clip(upper=9)
df['log_temp'] = np.log10(df['temp'].replace(0, np.nan))   # 3K..6.6M K -> ~3.5..6.8

# physics-informed coupling features (raw solar wind -> storm-driving quantities)
bt = df['b_magnitude']
clock = np.degrees(np.arctan2(df['by_gsm'], df['bz_gsm'])) % 360   # IMF clock angle, 0-360
south = np.sin(np.radians(clock) / 2) ** (8 / 3)                  # peaks at 180deg = due southward

df['newell'] = df['speed']**(4/3) * np.maximum(bt, 0)**(2/3) * south   # Newell coupling rate
df['rect_v'] = df['speed'] * np.clip(-df['bz_gsm'], 0, None)           # rectified solar wind speed
df['pressure'] = df['density'] * df['speed']**2                        # dynamic pressure proxy

print(len(df), 'hours:', df.index[0], '->', df.index[-1])

561024 hours: 1963-01-01 00:00:00 -> 2026-12-31 23:00:00


## Labels: Kp of the *next* 3-hour block

Kp is a 3-hourly index. Each hour gets the Kp of the next block boundary,
which makes the prediction lead time exact (always the upcoming block).

In [ ]:
kp = df['kp'].values
h = df.index.hour.values

next_off = (3 - (h % 3)) % 3
next_off[next_off == 0] = 3                    # boundary hours still predict the NEXT block
label_idx = np.arange(len(df)) + next_off

kp_next = np.full(len(df), np.nan)
ok = label_idx < len(df)
kp_next[ok] = kp[label_idx[ok]]
df['kp_next'] = kp_next

## Build windows: 24h x 5 features

Small gaps (<=3h) are linearly interpolated — solar wind varies slowly at
hourly scale, so this is safe and recovers the sparse pre-1995 coverage.
Windows with larger gaps are still dropped (strict).

In [ ]:
FEATS_A = ['bz_gsm', 'speed', 'density', 'newell', 'rect_v', 'pressure']
raw = pd.DataFrame(df[FEATS_A].values.astype(np.float32),
                   columns=FEATS_A, index=df.index)

data = raw.interpolate(limit=3, limit_direction='both').values.astype(np.float32)

W = sliding_window_view(data, (24, len(FEATS_A))).squeeze(1)   # (N-23, 24, 6)
y_a = kp_next[23:]
t_a = df.index[23:]

mask = ~np.isnan(W).any(axis=(1, 2)) & ~np.isnan(y_a)
X_a, y_a, t_a = W[mask], y_a[mask], t_a[mask]
print(f"windows: {len(X_a)}  (dropped {mask.size - mask.sum()} incomplete)")

windows: 375621  (dropped 185380 incomplete)


## Class mapping and time splits

0 = quiet (Kp<=3), 1 = active (Kp 4-5), 2 = storm (Kp>=6)

Time-ordered split - no shuffling. The solar ~11-year cycle means a random
split would let the model memorize cycle phase instead of learning physics,
and test scores would be optimistic. The test set models "the real future,"
which is exactly what the deployed board faces.

In [ ]:
def to_class(kp):
    return np.select([kp <= 3, kp <= 5], [0, 1], default=2)

tr = t_a <  '2011-01-01'
va = (t_a >= '2011-01-01') & (t_a < '2019-01-01')
te = t_a >= '2019-01-01'

XA = dict(Xtr=X_a[tr], Xva=X_a[va], Xte=X_a[te],
          ytr=to_class(y_a[tr]), yva=to_class(y_a[va]), yte=to_class(y_a[te]),
          ttr=t_a[tr], tva=t_a[va], tte=t_a[te])

for part in ['tr', 'va', 'te']:
    n = len(XA['y' + part])
    c = np.bincount(XA['y' + part], minlength=3) / n
    print(f"{part:3s}: n={n:6d}  quiet={c[0]:.2f}  active={c[1]:.2f}  storm={c[2]:.3f}")

tr : n=240312  quiet=0.77  active=0.20  storm=0.030
va : n= 69349  quiet=0.87  active=0.12  storm=0.015
te : n= 65960  quiet=0.85  active=0.13  storm=0.016


## Normalize (train statistics only) and save

Scaling uses train mean/std applied to all splits — computing them on the
full dataset would leak test information. Constants are saved to JSON
because the same numbers must be baked into the ESP32 firmware later.

In [ ]:
flat = XA['Xtr'].reshape(-1, len(FEATS_A))          # all train timesteps pooled
mu, sd = flat.mean(axis=0), flat.std(axis=0)
sd = np.where(sd < 1e-6, 1.0, sd)                    # guard against constant columns

def sc(x):
    return (x - mu) / sd

XA = {k: (sc(v) if k.startswith('X') else v) for k, v in XA.items()}

np.savez_compressed(f'{OUT_DIR}/windows_a.npz', **XA)
with open(f'{OUT_DIR}/windows_a_norm.json', 'w') as f:
    json.dump({'mean': mu.tolist(), 'std': sd.tolist(), 'features': FEATS_A}, f, indent=1)

print('saved windows_a.npz + windows_a_norm.json')

saved windows_a.npz + windows_a_norm.json


## TODO (notebook 02)
- Class-weighting decision based on storm-class percentages above
- Model candidates: window-stats MLP / 1D-CNN / GRU — compare on validation